In [ ]:
import wave
from pathlib import Path

import numpy as np
import pandas as pd
import tgt


# helpers

def load_table(path, participant_col=None, sheet_name=0):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path, dtype={participant_col: str} if participant_col else None)

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(
            path,
            sheet_name=sheet_name,
            dtype={participant_col: str} if participant_col else None,
            engine="openpyxl",
        )

    raise ValueError(f"Unsupported metadata format: {suffix}")


def get_wav_duration(file_path):
    try:
        with wave.open(str(file_path), "rb") as f:
            return f.getnframes() / float(f.getframerate())
    except Exception:
        return 0.0


def get_valid_wavs_with_textgrids(folder):
    return sorted([
        wav_path for wav_path in folder.glob("*.wav")
        if wav_path.with_suffix(".TextGrid").exists()
    ])


def count_words_in_textgrid(textgrid_path):
    try:
        tg = tgt.read_textgrid(str(textgrid_path))
        tier = tg.get_tier_by_name("words")
    except Exception:
        return 0

    count = 0
    for interval in tier.intervals:
        text = str(interval.text).strip().lower()
        if text and text not in {"sil", "sp", "spn", "<sil>", "silence"}:
            count += 1
    return count


def get_folder_stats(folder):
    valid_wavs = get_valid_wavs_with_textgrids(folder)

    total_duration = 0.0
    total_words = 0
    total_utterances = 0

    for wav_path in valid_wavs:
        tg_path = wav_path.with_suffix(".TextGrid")

        dur = get_wav_duration(wav_path)
        n_words = count_words_in_textgrid(tg_path)

        if dur > 0 and n_words > 0:
            total_duration += dur
            total_words += n_words
            total_utterances += 1

    return total_duration, total_words, total_utterances


# metadata

def normalize_gender(value):
    if pd.isna(value):
        return pd.NA

    s = str(value).strip().lower()
    return s if s else pd.NA


def format_psychosis_score(score_value, scale_name="PANSS10"):
    if pd.isna(score_value):
        return f"{scale_name} NA"

    score_value = float(score_value)
    if score_value.is_integer():
        return f"{scale_name} {int(score_value)}"
    return f"{scale_name} {score_value:.2f}".rstrip("0").rstrip(".")


def get_panss10_columns(metadata_df):
    expected = [
        "panss_p1_delusions",
        "panss_p2_disorg",
        "panss_p3_hallucination",
        "panss_n1_blunted",
        "panss_n4_apathy",
        "panss_n6_flow",
        "panss_g2_anxiety",
        "panss_g5_mannerisms",
        "panss_g6_depression",
        "panss_g9_unusual",
    ]

    missing = [c for c in expected if c not in metadata_df.columns]
    if missing:
        raise ValueError(f"Missing expected PANSS columns: {missing}")

    return expected


def compute_psychosis_columns(meta):
    meta = meta.copy()

    if "demo_sex" not in meta.columns:
        raise ValueError("Missing required column: demo_sex")
    if "demo_age_years" not in meta.columns:
        raise ValueError("Missing required column: demo_age_years")

    meta["label_psychosis"] = "Y"
    meta["gender"] = meta["demo_sex"].apply(normalize_gender)
    meta["age"] = pd.to_numeric(meta["demo_age_years"], errors="coerce")

    panss10_cols = get_panss10_columns(meta)
    panss10 = meta[panss10_cols].apply(pd.to_numeric, errors="coerce")

    meta["panss10_total_numeric"] = panss10.sum(axis=1, min_count=1)
    meta["psychosis_remission"] = panss10.le(3).all(axis=1).map({True: "Y", False: "N"})
    meta["score_psychosis"] = meta["panss10_total_numeric"].apply(
        lambda x: format_psychosis_score(x, "PANSS10")
    )

    return meta


def assign_psychosis_and_score(meta, participant_col):
    meta = compute_psychosis_columns(meta)

    meta[participant_col] = (
        meta[participant_col]
        .astype(str)
        .str.strip()
        .str.strip("'")
        .str.strip('"')
        .str.zfill(3)
    )

    result = meta[
        [
            participant_col,
            "label_psychosis",
            "score_psychosis",
            "psychosis_remission",
            "gender",
            "age",
        ]
    ].copy()

    result = result.dropna(subset=[participant_col])
    result = result.drop_duplicates(subset=[participant_col], keep="first")

    return dict(
        zip(
            result[participant_col],
            zip(
                result["label_psychosis"],
                result["score_psychosis"],
                result["psychosis_remission"],
                result["gender"],
                result["age"],
            ),
        )
    )


# split logic-

def assign_grouped_balanced_splits(
    df,
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    utterance_weight=14.0,
    word_weight=10.0,
    gender_weight=6.0,
    remission_weight=8.0,
    psychosis_weight=2.0,
    min_psy_groups_val=6,
    min_psy_groups_test=6,
    min_remission_y_groups_val=2,
    min_remission_y_groups_test=2,
    min_remission_n_groups_val=2,
    min_remission_n_groups_test=2,
    soft_cap_factor=1.20,
    overflow_penalty=1e6,
):
    splits = ["train", "val", "test"]
    ratios = {"train": train_ratio, "val": val_ratio, "test": test_ratio}

    gender_labels = sorted(df["gender"].dropna().unique())
    psy_labels = sorted(df["label_psychosis"].dropna().unique())
    rem_labels = sorted(df["psychosis_remission"].dropna().unique())

    speaker_groups = []

    for pid, subdf in df.groupby("participant_id"):
        total_utts = float(subdf["num_utterances"].sum())
        total_words = float(subdf["num_words"].sum())

        gender_counts = {g: 0.0 for g in gender_labels}
        psy_counts = {p: 0.0 for p in psy_labels}
        rem_counts = {r: 0.0 for r in rem_labels}

        for label, dsub in subdf.groupby("gender"):
            gender_counts[label] = float(dsub["num_utterances"].sum())

        for label, dsub in subdf.groupby("label_psychosis"):
            psy_counts[label] = float(dsub["num_utterances"].sum())

        for label, dsub in subdf.groupby("psychosis_remission"):
            rem_counts[label] = float(dsub["num_utterances"].sum())

        speaker_groups.append(
            {
                "participant_id": pid,
                "total_utterances": total_utts,
                "total_words": total_words,
                "gender_counts": gender_counts,
                "psy_counts": psy_counts,
                "rem_counts": rem_counts,
                "has_psy": bool((subdf["label_psychosis"] == "Y").any()),
                "has_rem_y": bool((subdf["psychosis_remission"] == "Y").any()),
                "has_rem_n": bool((subdf["psychosis_remission"] == "N").any()),
            }
        )

    speaker_groups.sort(key=lambda x: x["total_utterances"], reverse=True)

    total_utts = float(df["num_utterances"].sum())
    total_words = float(df["num_words"].sum())

    total_gender = {
        g: float(df.loc[df["gender"] == g, "num_utterances"].sum())
        for g in gender_labels
    }
    total_psy = {
        p: float(df.loc[df["label_psychosis"] == p, "num_utterances"].sum())
        for p in psy_labels
    }
    total_rem = {
        r: float(df.loc[df["psychosis_remission"] == r, "num_utterances"].sum())
        for r in rem_labels
    }

    target_utts = {s: ratios[s] * total_utts for s in splits}
    target_words = {s: ratios[s] * total_words for s in splits}

    target_gender = {
        s: {g: ratios[s] * total_gender[g] for g in gender_labels}
        for s in splits
    }
    target_psy = {
        s: {p: ratios[s] * total_psy[p] for p in psy_labels}
        for s in splits
    }
    target_rem = {
        s: {r: ratios[s] * total_rem[r] for r in rem_labels}
        for s in splits
    }

    current_utts = {s: 0.0 for s in splits}
    current_words = {s: 0.0 for s in splits}
    current_gender = {s: {g: 0.0 for g in gender_labels} for s in splits}
    current_psy = {s: {p: 0.0 for p in psy_labels} for s in splits}
    current_rem = {s: {r: 0.0 for r in rem_labels} for s in splits}

    psy_counts = {"val": 0, "test": 0}
    rem_y_counts = {"val": 0, "test": 0}
    rem_n_counts = {"val": 0, "test": 0}

    assignment = {}

    def over_soft_cap(split, speaker):
        if split == "train":
            return False
        after = current_utts[split] + speaker["total_utterances"]
        return after > soft_cap_factor * target_utts[split]

    def add_to_split(split, speaker):
        assignment[speaker["participant_id"]] = split

        current_utts[split] += speaker["total_utterances"]
        current_words[split] += speaker["total_words"]

        for g in gender_labels:
            current_gender[split][g] += speaker["gender_counts"].get(g, 0.0)

        for p in psy_labels:
            current_psy[split][p] += speaker["psy_counts"].get(p, 0.0)

        for r in rem_labels:
            current_rem[split][r] += speaker["rem_counts"].get(r, 0.0)

        if split in {"val", "test"}:
            if speaker["has_psy"]:
                psy_counts[split] += 1
            if speaker["has_rem_y"]:
                rem_y_counts[split] += 1
            if speaker["has_rem_n"]:
                rem_n_counts[split] += 1

    def candidate_cost(split, speaker):
        eps = 1e-8
        cost = 0.0

        after_utts = current_utts[split] + speaker["total_utterances"]
        after_words = current_words[split] + speaker["total_words"]

        if split != "train" and after_utts > soft_cap_factor * target_utts[split]:
            excess_ratio = after_utts / (target_utts[split] + eps)
            cost += overflow_penalty * (excess_ratio - soft_cap_factor)

        cost += utterance_weight * (((after_utts - target_utts[split]) / (target_utts[split] + eps)) ** 2)
        cost += word_weight * (((after_words - target_words[split]) / (target_words[split] + eps)) ** 2)

        for g in gender_labels:
            after = current_gender[split][g] + speaker["gender_counts"].get(g, 0.0)
            target = target_gender[split][g] + eps
            cost += gender_weight * (((after - target) / target) ** 2)

        for p in psy_labels:
            after = current_psy[split][p] + speaker["psy_counts"].get(p, 0.0)
            target = target_psy[split][p] + eps
            cost += psychosis_weight * (((after - target) / target) ** 2)

        for r in rem_labels:
            after = current_rem[split][r] + speaker["rem_counts"].get(r, 0.0)
            target = target_rem[split][r] + eps
            cost += remission_weight * (((after - target) / target) ** 2)

        return cost

    unassigned = []

    for speaker in speaker_groups:
        placed = False

        if speaker["has_psy"]:
            candidates = []
            if psy_counts["val"] < min_psy_groups_val and not over_soft_cap("val", speaker):
                candidates.append("val")
            if psy_counts["test"] < min_psy_groups_test and not over_soft_cap("test", speaker):
                candidates.append("test")

            if candidates:
                best = min(candidates, key=lambda s: candidate_cost(s, speaker))
                add_to_split(best, speaker)
                placed = True

        if not placed:
            unassigned.append(speaker)

    still_unassigned = []

    for speaker in unassigned:
        placed = False
        candidates = []

        if speaker["has_rem_y"]:
            if rem_y_counts["val"] < min_remission_y_groups_val and not over_soft_cap("val", speaker):
                candidates.append("val")
            if rem_y_counts["test"] < min_remission_y_groups_test and not over_soft_cap("test", speaker):
                candidates.append("test")

        if speaker["has_rem_n"]:
            if rem_n_counts["val"] < min_remission_n_groups_val and not over_soft_cap("val", speaker):
                candidates.append("val")
            if rem_n_counts["test"] < min_remission_n_groups_test and not over_soft_cap("test", speaker):
                candidates.append("test")

        candidates = sorted(set(candidates))

        if candidates:
            best = min(candidates, key=lambda s: candidate_cost(s, speaker))
            add_to_split(best, speaker)
            placed = True

        if not placed:
            still_unassigned.append(speaker)

    for speaker in still_unassigned:
        candidates = [
            s for s in splits
            if s == "train" or not over_soft_cap(s, speaker)
        ]

        if not candidates:
            candidates = ["train"]

        best = min(candidates, key=lambda s: candidate_cost(s, speaker))
        add_to_split(best, speaker)

    out = df.copy()
    out["split"] = out["participant_id"].map(assignment)

    return out


# main

def create_balanced_splits_tang(
    mfa_root,
    metadata_path,
    output_csv,
    participant_col,
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
):
    dataset_name = "TANG"

    meta = load_table(metadata_path, participant_col=participant_col)
    psychosis_map = assign_psychosis_and_score(meta, participant_col=participant_col)

    data = []
    root = Path(mfa_root)
    speaker_folders = [folder for folder in sorted(root.iterdir()) if folder.is_dir()]

    print(f"Scanning speakers... found {len(speaker_folders)} speaker folders")

    for i, folder in enumerate(speaker_folders, 1):
        pid = folder.name.strip().strip("'").strip('"').zfill(3)
        print(f"[{i}/{len(speaker_folders)}] {pid}")

        psychosis_info = psychosis_map.get(pid)

        if psychosis_info is None:
            print(f"  Skipping {pid}: no metadata match.")
            continue

        label_psychosis, score_psychosis, psychosis_remission, gender, age = psychosis_info

        if pd.isna(gender):
            print(f"  Skipping {pid}: missing gender.")
            continue

        duration, num_words, num_utterances = get_folder_stats(folder)

        if duration > 0 and num_words > 0 and num_utterances > 0:
            data.append(
                {
                    "participant_id": pid,
                    "dataset": dataset_name,
                    "base_pid": pid,
                    "gender": gender,
                    "age": age,
                    "label_depression": pd.NA,
                    "score_depression": pd.NA,
                    "depression_severity": pd.NA,
                    "label_psychosis": label_psychosis,
                    "psychosis_remission": psychosis_remission,
                    "score_psychosis": score_psychosis,
                    "duration": duration,
                    "num_words": int(num_words),
                    "num_utterances": int(num_utterances),
                }
            )
            print(
                f"  Added {pid}: utterances={num_utterances}, "
                f"words={num_words}, duration={duration:.2f}s, remission={psychosis_remission}"
            )
        else:
            print(f"  Skipping {pid}: no usable wav/TextGrid words.")

    df = pd.DataFrame(data)

    if df.empty:
        raise ValueError("No usable speaker folders found.")

    out_df = assign_grouped_balanced_splits(
        df,
        train_ratio=train_ratio,
        val_ratio=val_ratio,
        test_ratio=test_ratio,
    )

    out_df = out_df[
        [
            "participant_id",
            "base_pid",
            "dataset",
            "split",
            "gender",
            "age",
            "label_depression",
            "score_depression",
            "depression_severity",
            "label_psychosis",
            "psychosis_remission",
            "score_psychosis",
            "duration",
            "num_words",
            "num_utterances",
        ]
    ].sort_values(["split", "participant_id"])

    out_df.to_csv(output_csv, index=False)

    print("\nDone.")
    print(f"Saved to: {output_csv}")

    print("\nParticipants by split:")
    print(out_df.groupby("split")["participant_id"].nunique())

    print("\nUtterances by split:")
    print(out_df.groupby("split")["num_utterances"].sum())

    print("\nWords by split:")
    print(out_df.groupby("split")["num_words"].sum())

    print("\nGender by split:")
    print(out_df.groupby(["split", "gender"])["participant_id"].nunique())

    print("\nPsychosis remission by split:")
    print(out_df.groupby(["split", "psychosis_remission"])["participant_id"].nunique())

    print("\nUtterances by split and remission:")
    print(out_df.groupby(["split", "psychosis_remission"])["num_utterances"].sum())

    print("\nWords by split and remission:")
    print(out_df.groupby(["split", "psychosis_remission"])["num_words"].sum())

    print("\nAge summary by split:")
    print(
        out_df.assign(age_numeric=pd.to_numeric(out_df["age"], errors="coerce"))
        .groupby("split")["age_numeric"]
        .agg(["count", "min", "max", "mean", "median"])
        .round(2)
    )

    return out_df

In [ ]:
create_balanced_splits_tang(
    mfa_root="/work/DISCOURSE/Data/Discourse/AUDIO_CHUNKED/TANG/",
    metadata_path="/work/DISCOURSE/Data/Discourse/METADATA/TANG Metadata_x.xlsx",
    output_csv="TANG_splits.csv",
    participant_col="grid")